# Direct Research — Generación de Knowledge Base

Enfoque simplificado: una sola llamada a Gemini con instrucciones de investigación libre.
Sin pipeline de 4 pasos. Sin validator que filtre experiencias.

In [25]:
import sys
sys.path.append('../')
import json
from src.core.gemini_processor import GeminiProcessor
from src.core.prompt_utils import read_prompt_from_file, replace_variables, validate_prompt_variables
from src.core.base_knowledge import format_technical_specs
from src.sources.inventory.app import Inventory
from src.config.settings import COUNTRY

db_loader = Inventory()
gemini = GeminiProcessor()

## 1. Cargar base de datos

In [26]:
df_data_base = db_loader.load_db_from_country_selected()

MX


## 2. Seleccionar modelo

Cambiar el `code` según el modelo a procesar.

In [27]:
df_modelo = df_data_base[df_data_base["code"] == "MX2773-cflite-250-nk-carburada"]
df_modelo

,date,code,brand,model,year,title,type,technical_specs,publication_url,publication_image_url
429,05/05/2026,MX2773-cflite-250-nk-carburada,CFLITE,250 NK Carburada,2025,CFLITE 250 NK Carburada,Naked,"[{'key': 'ignition', 'value': 'Digital', 'type...",https://www.galgo.com/mx/motos/MX2773-cflite-2...,https://images.ctfassets.net/8zlbnewncp6f/01E0...


## 3. Extraer variables del modelo

In [28]:
for index, row in df_modelo.iterrows():
    BRAND = row["brand"]
    MODEL = row["model"]
    TITLE = row["title"]
    YEAR = "2025"
    TYPE = row["type"]
    PAIS = {"CO": "Colombia", "MX": "Mexico", "CL": "Chile"}.get(COUNTRY)
    TECHNICAL_SPECS = row["technical_specs"]
    CODE = row["code"]

print(f"Marca: {BRAND}")
print(f"Modelo: {MODEL}")
print(f"Title: {TITLE}")
print(f"Año: {YEAR}")
print(f"País: {PAIS}")
print(f"Tipo: {TYPE}")
print(f"Code: {CODE}")

Marca: CFLITE
Modelo: 250 NK Carburada
Title: CFLITE 250 NK Carburada
Año: 2025
País: Mexico
Tipo: Naked
Code: MX2773-cflite-250-nk-carburada


## 4. Construir prompt

In [29]:
PROMPT_PATH = "../src/data/input/prompts/direct_research_template.md"

ficha_formateada = format_technical_specs(TECHNICAL_SPECS)

prompt = replace_variables(read_prompt_from_file(PROMPT_PATH), {
    "{MARCA}": BRAND,
    "{MODELO}": MODEL,
    "{AÑO}": str(YEAR),
    "{PAIS}": PAIS,
    "{TIPO}": TYPE,
    "{TITLE}": TITLE,
    "{FICHA TECNICA}": ficha_formateada,
})

# Validar que no queden placeholders sin reemplazar
info = validate_prompt_variables(prompt)
if not info["valid"]:
    raise ValueError(f"Variables sin reemplazar: {info['missing_variables']}")

print(prompt)

# ROL

Eres un investigador especializado en experiencias reales de motocicletas en mercados latinoamericanos.
Tu trabajo es investigar a fondo la CFLITE 250 NK Carburada en Mexico y generar una base de conocimiento
experiencial basada en lo que reportan propietarios reales: cómo se siente, qué falla, qué enamora,
qué decepciona.

Tu enfoque es vivencial y sentimental: cómo se siente VIVIR con la moto, NO describir fichas técnicas.
Menciona conceptos técnicos SOLO cuando expliquen sensaciones, problemas o decisiones de compra.

---

# PROCESO DE INVESTIGACIÓN

Antes de generar el reporte, realiza búsquedas activas en este orden. Usa los resultados de TODAS
las búsquedas para construir el reporte:

1. "CFLITE 250 NK Carburada Mexico opiniones usuarios"
2. "CFLITE 250 NK Carburada Mexico experiencia propietario"
3. "CFLITE 250 NK Carburada Mexico problemas fallas"
4. "CFLITE 250 NK Carburada Mexico foro review"
5. "CFLITE 250 NK Carburada vs competidores Mexico"
6. "CFLITE 250 NK Carbura

## 5. Ejecutar investigación

In [30]:
knowledge_base = gemini.send_prompt(prompt)

METADATOS_BLOQUE = f"""[METADATOS]

Marca: {BRAND}
Modelo: {MODEL}
Año: {YEAR}
País: {PAIS}
Código publicación: {CODE}
"""

knowledge_base_completo = f"{METADATOS_BLOQUE}\n{knowledge_base}"
print(knowledge_base_completo)

[METADATOS]

Marca: CFLITE
Modelo: 250 NK Carburada
Año: 2025
País: Mexico
Código publicación: MX2773-cflite-250-nk-carburada

[SEGMENTO]

Coincide con tipo declarado

[SENTIMIENTO]

La CFLITE 250 NK Carburada genera un sentimiento muy positivo por democratizar el diseño y la potencia del motor monocilíndrico de CFMOTO a un precio accesible. Sin embargo, hay reservas sobre la pérdida de inyección electrónica, ABS y horquilla invertida, además de quejas recurrentes sobre la incomodidad para el pasajero y vibraciones a altas velocidades.

[SENSACIONES]

Estabilidad: Muy buena en ciudad gracias a su bajo peso y chasis ágil, pero se percibe inestable y nerviosa al superar los 130 km/h en carretera.
Vibraciones: Tolerables en bajas revoluciones, pero muy notorias a partir de los 90 km/h, afectando principalmente los posapiés y nublando por completo la visibilidad en los espejos retrovisores.
Frenado: Efectivo y con buen tacto inicial gracias a sus discos, pero los usuarios resienten la falt

## 6. Guardar output

In [31]:
nombre_archivo = f'../src/data/output/KB/{COUNTRY}-{BRAND}_{MODEL}.md'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    f.write(knowledge_base_completo)
print(f"Guardado en: {nombre_archivo}")

Guardado en: ../src/data/output/KB/MX-CFLITE_250 NK Carburada.md
